In [62]:
import pandas as pd 
import numpy as np

In [63]:
df = pd.read_csv('https://raw.githubusercontent.com/doeungim/ADP_2026/refs/heads/main/ADP_DATA/q1_subway_long.csv')
df

,날짜,호선,역번호,역명,구분,시간대,이용객수
0,2024-01-02,1,1001,서울역,승차,4,3507
1,2024-01-02,1,1001,서울역,승차,6,1686
2,2024-01-02,1,1001,서울역,승차,8,3746
3,2024-01-02,1,1001,서울역,승차,10,3049
4,2024-01-02,1,1001,서울역,승차,12,3202
...,...,...,...,...,...,...,...
275,2024-01-03,1,1098,임시역B,하차,14,0
276,2024-01-03,1,1098,임시역B,하차,16,0
277,2024-01-03,1,1098,임시역B,하차,18,0
278,2024-01-03,1,1098,임시역B,하차,20,0


In [64]:
df['구분시간대'] = df['구분'].astype(str) + '_' + df['시간대'].astype(str).str.zfill(2)
pivot_df = df.pivot(index = ['날짜','호선','역번호','역명'] , columns = '구분시간대', values = '이용객수').reset_index()

# ② 승차 전체 0 AND 하차 전체 0인 행 제거
pivot_df['승차합'] = pivot_df.iloc[:,4:14].sum(axis = 1)
pivot_df['하차합'] = pivot_df.iloc[:,15:24].sum(axis = 1) 

filter_df = pivot_df[(pivot_df['승차합'] != 0) | (pivot_df['하차합'] != 0)]
filter_df.head()

구분시간대,날짜,호선,역번호,역명,승차_04,승차_06,승차_08,승차_10,승차_12,승차_14,...,하차_08,하차_10,하차_12,하차_14,하차_16,하차_18,하차_20,하차_22,승차합,하차합
0,2024-01-02,1,1001,서울역,3507,1686,3746,3049,3202,748,...,3524,5807,5299,5222,1757,2827,5841,1783,32817,37965
1,2024-01-02,1,1002,시청,1844,709,2861,1419,2036,5680,...,3026,2009,5809,2435,2969,4786,1852,5584,22759,29008
2,2024-01-02,1,1003,종각,1204,4549,3241,1746,4041,4773,...,303,3233,2607,5642,148,2250,3011,5023,33068,26190
3,2024-01-02,1,1004,종로3가,3154,2785,4605,1004,4615,3496,...,3404,275,4943,1009,5650,4598,3117,865,26595,29806
4,2024-01-02,1,1005,동대문,5138,1189,240,2350,1030,3025,...,5904,619,5492,1945,2783,3913,4771,4064,21204,31357


In [65]:
# peak_ratio 생성: (승차_07 + 승차_08 + 승차_17 + 승차_18) / 승차 전체 합계
cols = ['승차_06','승차_08','승차_16','승차_18'] 

cols_sum = filter_df[cols].sum(axis = 1)
total = filter_df['승차합']
filter_df['peak_ratio'] = cols_sum / total
filter_df.head(5)


C:\Users\i2max-DoeunKim\AppData\Local\Temp\ipykernel_16744\2063614372.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filter_df['peak_ratio'] = cols_sum / total


구분시간대,날짜,호선,역번호,역명,승차_04,승차_06,승차_08,승차_10,승차_12,승차_14,...,하차_10,하차_12,하차_14,하차_16,하차_18,하차_20,하차_22,승차합,하차합,peak_ratio
0,2024-01-02,1,1001,서울역,3507,1686,3746,3049,3202,748,...,5807,5299,5222,1757,2827,5841,1783,32817,37965,0.359235
1,2024-01-02,1,1002,시청,1844,709,2861,1419,2036,5680,...,2009,5809,2435,2969,4786,1852,5584,22759,29008,0.309108
2,2024-01-02,1,1003,종각,1204,4549,3241,1746,4041,4773,...,3233,2607,5642,148,2250,3011,5023,33068,26190,0.430023
3,2024-01-02,1,1004,종로3가,3154,2785,4605,1004,4615,3496,...,275,4943,1009,5650,4598,3117,865,26595,29806,0.384584
4,2024-01-02,1,1005,동대문,5138,1189,240,2350,1030,3025,...,619,5492,1945,2783,3913,4771,4064,21204,31357,0.284852


In [66]:
# ④ peak_ratio 기준 상위 3개 역(날짜별) 출력 
filter_df.sort_values(by=['날짜', 'peak_ratio'], ascending = (True,False)).groupby('날짜').head(3)[['날짜','역명','peak_ratio']].reset_index(drop = True)


구분시간대,날짜,역명,peak_ratio
0,2024-01-02,종각,0.430023
1,2024-01-02,종로3가,0.384584
2,2024-01-02,서울역,0.359235
3,2024-01-03,종로3가,0.516234
4,2024-01-03,종각,0.473644
5,2024-01-03,시청,0.434666


In [67]:
df = pd.read_csv('https://raw.githubusercontent.com/doeungim/ADP_2026/refs/heads/main/ADP_DATA/q2_neonatal.csv')
df

,id,bwt,gest,pvh,ivh,ipe,sex,twn,lol,dob
0,1,1813.0,33.0,absent,possible,positive,1,1,3.0,2022-01-01
1,2,636.0,35.0,absent,NaN,definite,1,0,3.0,2022-01-04
2,3,1992.0,24.0,absent,absent,NaN,1,1,2.0,2022-01-07
3,4,878.0,35.0,absent,absent,positive,2,1,NaN,2022-01-10
4,5,816.0,32.0,possible,absent,absent,1,0,1.0,2022-01-13
...,...,...,...,...,...,...,...,...,...,...
115,116,1772.0,35.0,absent,positive,absent,1,0,3.0,2022-12-12
116,117,679.0,28.0,positive,definite,absent,2,0,1.0,2022-12-15
117,118,1485.0,27.0,positive,possible,definite,2,0,1.0,2022-12-18
118,119,1836.0,31.0,definite,absent,absent,1,0,4.0,2022-12-21


In [68]:
# Bleed: pvh·ivh·ipe 중 하나라도 'positive' 또는 'definite' → 1 
# 셋 모두 'absent' → 0 / 그 외 → NaN

def bleed(x) : 
    if x[['pvh','ivh','ipe']].isin(['positive','definite']).any() : 
        return 1 
    elif x[['pvh','ivh','ipe']].isin(['absent']).all() : 
        return 0 
    else : 
        np.nan

df['bleed'] = df.apply(bleed, axis =1 ) 
df['bleed'].value_counts()
#Ind: pvh·ivh·ipe 제외 컬럼(id 제외)에 결측 없으면 → 0
#위 조건 충족 + Bleed가 NaN → 1 / 나머지 → 2
#검증: Bleed 별 (0/1/NaN) 행 수, Ind 별 (0/1/2) 행 수를 value_counts(dropna=False)로 출력
#추가: Bleed==1인 행 중 bwt(출생체중) 평균과 Bleed==0인 행 중 bwt 평균을 비교 출력
df.head()

,id,bwt,gest,pvh,ivh,ipe,sex,twn,lol,dob,bleed
0,1,1813.0,33.0,absent,possible,positive,1,1,3.0,2022-01-01,1.0
1,2,636.0,35.0,absent,NaN,definite,1,0,3.0,2022-01-04,1.0
2,3,1992.0,24.0,absent,absent,NaN,1,1,2.0,2022-01-07,NaN
3,4,878.0,35.0,absent,absent,positive,2,1,NaN,2022-01-10,1.0
4,5,816.0,32.0,possible,absent,absent,1,0,1.0,2022-01-13,NaN


In [69]:
#Ind: pvh·ivh·ipe 제외 컬럼(id 제외)에 결측 없으면 → 0 -> bleed에도 
# 위 조건 충족 + Bleed가 NaN → 1 / 나머지 → 2
bleed_cols = ['pvh','ivh','ipe','id']
other_cols = df.columns.difference(bleed_cols)

def ind(x):
    if x[other_cols].notna().all():
        return 0
    elif x[other_cols].notna().all() and pd.isna(x['bleed']):
        return 1
    else:
        return 2

df.apply(ind, axis = 1).value_counts()
df['Ind'] = df.apply(ind, axis = 1)

In [70]:
# 3. 항공편 
df = pd.read_csv('https://raw.githubusercontent.com/doeungim/ADP_2026/refs/heads/main/ADP_DATA/q3_flights.csv')
df.head()

,flight_id,carrier,dep_delay,arr_delay,cancelled,cancellation_code,distance,air_time
0,FL0001,LJ,-99.4,-95.9,1.0,D,2377,35.0
1,FL0002,LJ,5.6,19.5,0.0,NaN,218,87.0
2,FL0003,BX,-57.9,-70.2,0.0,NaN,2931,354.0
3,FL0004,BX,-2.4,5.5,0.0,NaN,2612,347.0
4,FL0005,BX,4.4,3.8,0.0,NaN,2251,198.0


1. 아래 3가지 정합성 오류 행을 각각 찾아 건수와 flight_id를 출력하라
- 오류A: cancelled == 1 인데 dep_delay 또는 arr_delay 값이 존재
- 오류B: cancelled == 0 인데 dep_delay 가 NaN
- 오류C: cancelled 자체가 NaN

2. 3가지 오류 행을 모두 제거한 df_clean 생성
3. df_clean에서 carrier별 dep_delay 평균과 결측률을 함께 출력
4. speed_kmh = distance × 1.609 / (air_time / 60) 계산 후 1500 초과는 이상치로 NaN 처리, carrier별 중앙값 출력

In [71]:

# True / False로만 나오게 df[]쓰지 않음
e1 = (df['cancelled'] == 1) & (df[['dep_delay','arr_delay']].notna().any(axis = 1))
e2 = (df['cancelled'] == 0) & (df['dep_delay'].isna())
e3 = df['cancelled'].isna() 

df_clean = df[~ (e1 | e2 | e3)]

# 3. 
df_clean.groupby('carrier')['dep_delay'].agg(
    평균 = lambda x : x.mean().round(2) , 
    결측율 = lambda x : x.isna().mean().round(4)
)

# 4. 
speed_kmh = (df_clean['distance'] * 1.609 )/ (df_clean['air_time'] / 60)


df_clean['speed_kmh'] = np.where(
    (df_clean['air_time'].isna()) | (df_clean['air_time'] == 0) | (speed_kmh > 1500) , np.nan, speed_kmh.round(1)
)

df_clean.groupby('carrier')['speed_kmh'].median().round(1)


C:\Users\i2max-DoeunKim\AppData\Local\Temp\ipykernel_16744\3616504382.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['speed_kmh'] = np.where(


carrier
BX    823.6
KE    465.0
LJ    726.4
OZ    757.2
TW    694.2
Name: speed_kmh, dtype: float64

1. 기온·강수량: 관측소별로 독립 처리

2. 직전·직후 모두 존재 → (직전 + 직후) / 2
- 직후만 존재(앞쪽 결측) → 직후값
- 직전만 존재(뒤쪽 결측) → 직전값

3. 습도: 관측소별 중앙값 대체
4. 기온_7MA: 관측소별 7일 이동평균 컬럼 추가 (min_periods=1)
- 파생변수 precip_type: 강수량==0 → 'none' / 강수량>0 & 기온≤0 → 'snow' / 강수량>0 & 기온>0 → 'rain' / NaN → 'unknown'
5. 처리 후 결측 수 0임을 확인 + 관측소별 기온 평균, precip_type 분포 출력

In [114]:
w = pd.read_csv('https://raw.githubusercontent.com/doeungim/ADP_2026/refs/heads/main/ADP_DATA/q4_weather.csv')
w.head()

,날짜,관측소,기온,강수량,풍속,습도
0,2024-01-01,AWS001,NaN,4.4,4.7,NaN
1,2024-01-02,AWS001,NaN,0.6,3.3,39.0
2,2024-01-03,AWS001,NaN,NaN,4.7,90.0
3,2024-01-04,AWS001,-12.7,5.3,3.6,53.0
4,2024-01-05,AWS001,-9.0,0.8,5.0,50.0


In [115]:
w['날짜'] = pd.to_datetime(w['날짜'])

In [119]:
# 직전·직후 모두 존재 → (직전 + 직후) / 2
# 직후만 존재(앞쪽 결측) → 직후값
# 직전만 존재(뒤쪽 결측) → 직전값

def fill_col(df, col) : 
    ff = df.groupby('관측소')[col].transform(lambda x : x.ffill())
    bf = df.groupby('관측소')[col].transform(lambda x : x.bfill()) 

    # True/False로 나옴 
    na = df[col].isna()

    filled = np.select(
        [na & ff.notna() & bf.notna() , 
         na & ff.isna() & bf.notna() , 
         na & ff.notna() & bf.isna()        
        ],  [(ff + bf) /2 , bf , ff] , default = np.nan 
    )

    return pd.array(np.where(na, filled, df[col]), dtype='float64') 

w['기온']   = fill_col(w, '기온')
w['강수량'] = fill_col(w, '강수량') 

In [120]:
# 습도: 관측소별 중앙값 대체
w['습도'] = w.groupby('관측소')['습도'].transform(lambda x : x.fillna(x.median()))

In [122]:
# ── 3. 기온 7일 이동평균 ──────────────────────────────────────
w['기온_7MA'] = w.groupby('관측소')['기온'].transform(
    lambda x : x.rolling(7, min_periods = 1).mean()
)

In [125]:
# -- 4 . 파생변수 precip_type: 강수량==0 → 'none' / 강수량>0 & 기온≤0 → 'snow' / 강수량>0 & 기온>0 → 'rain' / NaN → 'unknown'

w['precip_type'] = np.select(
    [w['강수량'].isna() , #조건1
    w['강수량'] == 0 ,    #조건2
    (w['강수량'] > 0) & (w['기온'] <= 0) , #조건3 
    (w['강수량'] > 0) & (w['기온'] > 0)] , #조건4
    ['unknown' ,'none','snow','rain'] , #대응값
    default = 'unknown'
)

##### **Q5. q5_sales.csv — IQR 이상치 처리 + 계층적 결측 대체 + 순매출 ★★★★☆**
처리 조건
1. 판매량: 제품코드별 IQR 기준 이상치(Q1-1.5×IQR 미만 or Q3+1.5×IQR 초과)를 NaN으로 변환 후 제품별 중앙값으로 대체. 처리 전 이상치 건수를 제품별로 출력
2. 단가: 제품코드 × 연도 그룹 중앙값 → 남은 NA는 제품코드 전체 중앙값 (2단계)
3. 반품수량: NaN → 0 (미반품으로 간주)
4. 순매출 = (판매량 - 반품수량) × 단가 컬럼 생성
5. 제품별 이상치 비율, 순매출 합계 출력

In [126]:
df = pd.read_csv('https://raw.githubusercontent.com/doeungim/ADP_2026/refs/heads/main/ADP_DATA/q5_sales.csv')
df.head()

,날짜,제품코드,판매량,단가,반품수량
0,2023-01-01,PRD_A,126,48340.0,NaN
1,2023-01-02,PRD_A,210,31828.0,15.0
2,2023-01-03,PRD_A,82,32692.0,8.0
3,2023-01-04,PRD_A,179,32331.0,4.0
4,2023-01-05,PRD_A,192,25518.0,3.0


In [127]:
df['날짜'] = pd.to_datetime(df['날짜'])
df['연도'] = df['날짜'].dt.year

In [137]:
grp = df.groupby('제품코드')['판매량']

q1 = grp.transform(lambda x : x.quantile(0.25))
q3 = grp.transform(lambda x : x.quantile(0.75))
iqr = q3 - q1 
mask = (df['판매량'] < q1 - 1.5 * iqr) | (df['판매량'] > q3 + 1.5 * iqr)

df.loc[mask, '판매량'] = np.nan

# 제품별 중앙값으로 대체. 처리 전 이상치 건수를 제품별로 출력
df['판매량'] = df.groupby('제품코드')['판매량'].transform(
                                                            lambda x : x.fillna(x.median())
)

In [139]:
#단가: 제품코드 × 연도 그룹 중앙값 → 남은 NA는 제품코드 전체 중앙값 (2단계)
med_gy = df.groupby(['제품코드','연도'])['단가'].transform('median') 

df['단가'] = df['단가'].fillna(med_gy)

df['단가'] = df.groupby('제품코드')['단가'].transform(
                                                       lambda x : x.fillna(x.median())
)

In [145]:
#반품수량: NaN → 0 (미반품으로 간주)
df['반품수량'] = df['반품수량'].apply(lambda x : 0 if pd.isna(x) else x)

#순매출 = (판매량 - 반품수량) × 단가 컬럼 생성
df['순매출'] = (df['판매량'] - df['반품수량']) * df['단가']


##### **Q6. q6_rfm.csv — RFM 등급 파생 + 결측 조건 분기 대체 ★★★★☆**
1. R·F·M 각각 qcut 4분위 점수 부여 (R은 낮을수록 좋으므로 역순: labels=[4,3,2,1])
2. R·F·M 중 하나라도 NaN → RFM_score = NaN, 그 외 → R+F+M 합산
3. segment: score≥10 → Champions / 7~9 → Loyal / 4~6 → AtRisk / ≤3 → Lost / NaN → NaN
4. channel 결측 처리: vip_flag==1 → 'vip_direct' / vip_flag==0 → 전체 최빈값 / vip_flag도 NaN → 'unknown'
5. vip_flag 결측: channel별 최빈값으로 대체
6. segment별 고객 수·평균 monetary 출력

In [166]:
df = pd.read_csv('https://raw.githubusercontent.com/doeungim/ADP_2026/refs/heads/main/ADP_DATA/q6_rfm.csv')
df.head()

,customer_id,recency_days,frequency,monetary,join_year,channel,vip_flag
0,C00001,129.0,29.0,NaN,2022,offline,0.0
1,C00002,171.0,79.0,3570873.0,2023,online,0.0
2,C00003,27.0,70.0,4563906.0,2022,online,1.0
3,C00004,199.0,48.0,2607804.0,2019,online,0.0
4,C00005,314.0,15.0,1375340.0,2021,app,0.0


In [167]:
df['R'] = pd.qcut(df['recency_days'] , 4, labels = [4,3,2,1] , duplicates= 'drop').astype('float')
df['F'] = pd.qcut(df['frequency'] , 4, labels = [4,3,2,1] , duplicates = 'drop').astype('float')
df['M'] = pd.qcut(df['monetary'], 4, labels = [4,3,2,1], duplicates ='drop').astype('float')
df.head()

,customer_id,recency_days,frequency,monetary,join_year,channel,vip_flag,R,F,M
0,C00001,129.0,29.0,NaN,2022,offline,0.0,3.0,3.0,NaN
1,C00002,171.0,79.0,3570873.0,2023,online,0.0,2.0,1.0,2.0
2,C00003,27.0,70.0,4563906.0,2022,online,1.0,4.0,2.0,1.0
3,C00004,199.0,48.0,2607804.0,2019,online,0.0,2.0,3.0,2.0
4,C00005,314.0,15.0,1375340.0,2021,app,0.0,1.0,4.0,3.0


In [168]:
#R·F·M 중 하나라도 NaN → RFM_score = NaN, 그 외 → R+F+M 합산
def rfm(x) : 
    if pd.isna(x[['R','F','M']]).any() : 
        return np.nan
    else : 
        return x['R'] + x['F'] + x['M']

df.apply(rfm, axis = 1)

rfm_na = df[['R','F','M']].isna().any(axis=1)
df['RFM_score'] = np.where(rfm_na, np.nan, df['R'] + df['F'] + df['M'])


In [169]:
# Segment 
def segment(x) : 
    if pd.isna(x) : 
        return np.nan 
    elif x >= 10 : 
        return 'Champion' 
    elif x >= 7 : 
        return 'Loyal'
    elif x >= 4 : 
        return 'AtRisk' 
    else : 
        return 'Lost'

df['segment'] = df['RFM_score'].apply(segment)

In [170]:
#channel 결측 처리: vip_flag==1 → 'vip_direct' / vip_flag==0 → 전체 최빈값 / vip_flag도 NaN → 'unknown'

# 1. Channel이 결측인 곳 
channel_na = pd.isna(df['channel'])

df.loc[channel_na & df['vip_flag'] == 1, 'channel']  == 'vip_direct'
df.loc[channel_na & df['vip_flag'] == 0, 'channel'] == df['channel'].mode()[0]
df.loc[channel_na & pd.isna(df['vip_flag']), 'channel'] == 'unknown'


Series([], Name: channel, dtype: bool)

In [176]:
# ── 5. vip_flag 결측: channel별 최빈값 ───────────────────────
df['vip_flag'] = df.groupby('channel')['vip_flag'].transform(
    lambda x: x.fillna(x.mode()[0] if len(x.mode()) > 0 else 0)
)

##### **q7_clinical.csv — 임상 다중 조건 파생 + 그룹 기반 결측 대체  ★★★★☆**
1. hypertension: sbp 또는 dbp 중 하나라도 NA → NaN / sbp≥140 또는 dbp≥90 → 1 / 그 외 → 0
2.
  - diabetes_risk: glucose·hba1c 둘 다 NA → 'unknown'
  - 하나라도 ≥기준(glucose≥126 또는 hba1c≥6.5) → 'high'
  - 하나라도 전단계(glucose 100~125 또는 hba1c 5.7~6.4) → 'pre' / 그 외 → 'normal'
  - ⚠ 하나만 NA일 때 존재하는 값으로만 판단: .fillna(False) 패턴 사용
3. complete_case: age·sbp·dbp·bmi·glucose·hba1c·smoker 모두 존재 → 1, 아니면 0
4. 결측 대체: age·bmi → 전체 중앙값 / sbp·dbp·glucose·hba1c → diabetes_risk별 중앙값 / smoker → 최빈값

In [177]:
df = pd.read_csv('https://raw.githubusercontent.com/doeungim/ADP_2026/refs/heads/main/ADP_DATA/q7_clinical.csv')
df.head()

,patient_id,age,sbp,dbp,bmi,glucose,hba1c,smoker,diabetes_dx
0,P0001,28.0,107.0,104.0,22.9,289.0,5.9,1.0,0.0
1,P0002,NaN,122.0,110.0,29.7,219.0,8.8,0.0,1.0
2,P0003,32.0,156.0,81.0,NaN,353.0,10.4,0.0,0.0
3,P0004,55.0,126.0,95.0,20.1,308.0,4.3,NaN,0.0
4,P0005,18.0,128.0,85.0,21.7,332.0,NaN,0.0,1.0


In [188]:
# 1) mask 
def one(x) :
    s = x['sbp']
    d = x['dbp']

    if pd.isna(s) or pd.isna(d) : 
        return np.nan 

    elif s >= 140 or d >= 90 : 
        return 1
    else : 
        return 0 
print(df.apply(one, axis = 1).value_counts())

# 2) np.where
mask = df[['sbp','dbp']].isna().any(axis = 1)


df['hypertension'] = np.where(
                                mask, np.nan, 
                                ((df['sbp'] >= 140) | (df['dbp'] >= 90)).astype(int)
                            )


1.0    166
0.0     48
Name: count, dtype: int64


In [193]:
# 2.  diabetes_risk: glucose·hba1c 둘 다 NA → 'unknown'
#  - 하나라도 ≥기준(glucose≥126 또는 hba1c≥6.5) → 'high'
#  - 하나라도 전단계(glucose 100~125 또는 hba1c 5.7~6.4) → 'pre' / 그 외 → 'normal'
#   하나만 NA일 때 존재하는 값으로만 판단: .fillna(False) 패턴 사용

con1 = df[['glucose','hba1c']].isna().all(axis=1)
(df['glucose'] >= 126) | (df['hba1c'] >= 6.5)

0      True
1      True
2      True
3      True
4      True
       ... 
245    True
246    True
247    True
248    True
249    True
Length: 250, dtype: bool

In [192]:
df[['glucose','hba1c']].isna().all(axis=1)

0      False
1      False
2      False
3      False
4      False
       ...  
245    False
246    False
247    False
248    False
249    False
Length: 250, dtype: bool